In [1]:
import pandas as pd
import numpy as np

print("Downloading dataset...")
# 1. Download the dataset directly from the UCI Machine Learning Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/splice-junction-gene-sequences/splice.data"

# The raw data has no column names, so we assign them
columns = ['Class', 'Instance_Name', 'Sequence']
df = pd.read_csv(url, names=columns)

# 2. Clean the data (remove hidden spaces in the text)
df['Class'] = df['Class'].str.strip()
df['Sequence'] = df['Sequence'].str.strip()

# 3. Tokenize the DNA (Convert A, C, G, T into integers)
# Neural networks need numbers. We map A=1, C=2, G=3, T=4. 
# (Any weird ambiguous characters like 'N' get mapped to 0).
def dna_to_numbers(seq):
    mapping = {'A': 1, 'C': 2, 'G': 3, 'T': 4}
    return [mapping.get(char.upper(), 0) for char in seq]

df['Encoded_Sequence'] = df['Sequence'].apply(dna_to_numbers)

# 4. Encode the Target Labels
# 'EI' (Exon-Intron junction), 'IE' (Intron-Exon junction), or 'N' (Neither)
label_mapping = {'EI': 0, 'IE': 1, 'N': 2}
df['Target'] = df['Class'].map(label_mapping)

print("Data processing complete!")
# Let's look at the first 3 rows to see the transformation
df[['Class', 'Target', 'Sequence', 'Encoded_Sequence']].head(3)

Data processing complete!


,Class,Target,Sequence,Encoded_Sequence
0,EI,0,CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...,"[2, 2, 1, 3, 2, 4, 3, 2, 1, 4, 2, 1, 2, 1, 3, ..."
1,EI,0,AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...,"[1, 3, 1, 2, 2, 2, 3, 2, 2, 3, 3, 3, 1, 3, 3, ..."
2,EI,0,GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...,"[3, 1, 3, 3, 4, 3, 1, 1, 3, 3, 1, 2, 3, 4, 2, ..."


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Preparing data for the neural network...")
# 1. Format the data perfectly for TensorFlow
X = pad_sequences(df['Encoded_Sequence'].tolist(), padding='post')
y = np.array(df['Target'].tolist())

# 2. Split data: 80% for training the AI, 20% for testing it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")

print("\nBuilding the Lightweight BiLSTM Model...")
# 3. Build the Neural Network Architecture
model = Sequential([
    # Step A: Embedding layer turns our 1,2,3,4 numbers into rich mathematical vectors
    Embedding(input_dim=5, output_dim=16),
    
    # Step B: The BiLSTM reads the DNA sequence both forwards and backwards!
    # We keep it small (16 units) so it trains super fast on your CPU.
    Bidirectional(LSTM(16)),
    
    # Step C: Output layer has 3 neurons because we have 3 classes (EI, IE, Neither)
    Dense(3, activation='softmax')
])

# 4. Compile the model (telling it how to learn)
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Show a summary of the architecture
model.summary()

print("\nStarting Training (This will only take a couple of minutes)...")
# 5. Train the model! We do 10 epochs (10 passes over the data)
history = model.fit(X_train, y_train, 
                    epochs=10, 
                    batch_size=32, 
                    validation_data=(X_test, y_test))

# 6. Save the trained model to your 'models' folder for the RAG pipeline to use later
model.save('models/genomic_bilstm.keras')
print("\n✅ Model successfully trained and saved!")

Preparing data for the neural network...
Training data shape: (2552, 60)

Building the Lightweight BiLSTM Model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting Training (This will only take a couple of minutes)...
Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.5231 - loss: 1.0110 - val_accuracy: 0.5235 - val_loss: 0.9723
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.6172 - loss: 0.8491 - val_accuracy: 0.6050 - val_loss: 0.8118
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.6387 - loss: 0.7751 - val_accuracy: 0.6191 - val_loss: 0.7811
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.6689 - loss: 0.7393 - val_accuracy: 0.6818 - val_loss: 0.7439
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.6685 - loss: 0.7201 - val_accuracy: 0.6755 - val_loss: 0.7261
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.6881 - loss: 0.7006 - val_accuracy: 0.6865 - val_loss: 0.7219
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.6947 - loss: 0.6868 - val_accuracy: 0.7179 - val_loss: 0.7046
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/s

In [3]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout

print("Building the Advanced CNN-BiLSTM Hybrid Model...")

hybrid_model = Sequential([
    # 1. Richer Embedding (Give it slightly more brainpower)
    Embedding(input_dim=5, output_dim=32),
    
    # 2. The CNN Layer (Quickly finds genetic motifs like 'GT' or 'AG')
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2), # Compresses the sequence to make it faster!
    
    # 3. The BiLSTM Layer (Reads the whole sequence context)
    Bidirectional(LSTM(32)),
    
    # 4. Dropout (Prevents the model from just memorizing the data)
    Dropout(0.3),
    
    # 5. Output Layer
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

hybrid_model.compile(optimizer='adam', 
                     loss='sparse_categorical_crossentropy', 
                     metrics=['accuracy'])

hybrid_model.summary()

print("\nTraining the Hybrid Model...")
# Let's train it for 15 epochs this time since it learns better
hybrid_history = hybrid_model.fit(X_train, y_train, 
                                  epochs=15, 
                                  batch_size=32, 
                                  validation_data=(X_test, y_test))

# Save this much better model!
hybrid_model.save('models/genomic_hybrid_bilstm.keras')
print("\n✅ Hybrid Model successfully trained and saved!")

Building the Advanced CNN-BiLSTM Hybrid Model...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the Hybrid Model...
Epoch 1/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.5263 - loss: 1.0104 - val_accuracy: 0.5643 - val_loss: 0.9496
Epoch 2/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.6089 - loss: 0.8478 - val_accuracy: 0.6082 - val_loss: 0.8019
Epoch 3/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6469 - loss: 0.7410 - val_accuracy: 0.6223 - val_loss: 0.7920
Epoch 4/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6763 - loss: 0.7188 - val_accuracy: 0.6850 - val_loss: 0.6917
Epoch 5/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.7030 - loss: 0.6600 - val_accuracy: 0.7069 - val_loss: 0.6924
Epoch 6/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.7406 - loss: 0.6033 - val_accuracy: 0.7367 - val_loss: 0.6028
Epoch 7/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7880 - loss: 0.5188 - val_accuracy: 0.7680 - val_loss: 0.5680
Epoch 8/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8166 - loss: 0.4